# 06 – B2 (debiased): Qwen2.5 0.5B / 1.5B as the scorer, averaged over two option orders (Kaggle)

Same setup as notebook 05, but every question is asked **twice**: options in the original order and in the reversed order. Probabilities are mapped back to the option *names* and averaged, which removes first-/last-position (letter) bias. For 5-option questions (`tool`, `action`) only the first and last positions are balanced; the middle positions are not (limitation).

Output: `results_qwen0.5b_avg.csv`, `results_qwen1.5b_avg.csv` → `results/baselines/`. Analysis:
`python scripts/analyze_baselines.py --qwen05 results/baselines/results_qwen0.5b_avg.csv --qwen15 results/baselines/results_qwen1.5b_avg.csv --out results/baselines/summary_val_avg.csv`

Notebook 05 (single order) is kept unchanged: it produced `results_qwen*.csv`.

Kaggle: GPU on, Internet on, one GPU session at a time. If the NaN error appears, change `dtype=torch.float16` to `torch.float32` in `run_model`.

In [ ]:
!pip install -q -U transformers accelerate

In [ ]:
import glob, json, os, time
import pandas as pd

xlsx = glob.glob("/kaggle/input/**/AISIS_DevSet_v1.xlsx", recursive=True)[0]
qjson = glob.glob("/kaggle/input/**/gate_questions_v2.json", recursive=True)[0]
dev = pd.read_excel(xlsx, sheet_name="DevSet").fillna("")
QUESTIONS = json.load(open(qjson, encoding="utf-8"))["questions"]
TOOLS = {k: v for k, v in QUESTIONS["tool"]["criteria"].items() if k != "none"}

# identical instance construction to notebook 01 (d1 gate): standard + dialect (+ wrong_passage for answerable)
instances = []
for r in dev.to_dict("records"):
    base = dict(dev_id=r["dev_id"], type=r["type"], topic=r["topic"],
                gold_action=r["label_action"], gold_supported=r["label_supported"],
                gold_specific=r["label_specific"], gold_emergency=r["label_emergency"],
                gold_tool=r["label_tool"])
    instances.append({**base, "version": "standard", "query": r["query_standard"], "passage": r["passage_text"]})
    instances.append({**base, "version": "dialect", "query": r["query_dialect_synth"], "passage": r["passage_text"]})
    if r["type"] == "answerable":
        instances.append({**base, "version": "wrong_passage", "query": r["query_standard"],
                          "passage": r["wrong_passage_text"],
                          "gold_action": "cannot_answer", "gold_supported": "not_supported"})
print(len(instances), "instances")  # expected 960

In [ ]:
import gc, torch, string
from transformers import AutoTokenizer, AutoModelForCausalLM

SYSTEM = ("You are a careful assistant for rural Bangladeshi users. You will be given a user query "
          "(Bengali, possibly dialectal), a retrieved passage and a multiple-choice question. "
          "Answer with the single letter of the best option and nothing else.")

def build_prompt(inst, qname, reverse=False):
    q = QUESTIONS[qname]
    keys = list(q["criteria"].keys())
    if reverse:
        keys = keys[::-1]                      # same options, opposite order
    letters = string.ascii_uppercase[:len(keys)]
    opts = "\n".join(f"{l}. {k}: {q['criteria'][k]}" for l, k in zip(letters, keys))
    tools = "\n".join(f"- {k}: {v}" for k, v in TOOLS.items())
    user = (f"User query: {inst['query']}\n\nRetrieved passage: {inst['passage']}\n\n"
            f"Available tools:\n{tools}\n\nQuestion: {q['instructions']}\n\nOptions:\n{opts}\n\n"
            "Answer with only the letter.")
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}], keys, letters

def letter_probs(tok, model, messages, keys, letters):
    enc = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        logits = model(**enc).logits[0, -1].float()
    ids = [tok.encode(l, add_special_tokens=False)[0] for l in letters]
    p = torch.softmax(logits[ids], dim=-1).cpu().tolist()
    return dict(zip(keys, p))                  # keyed by option NAME, so both orders line up

def avg_probs(tok, model, inst, qname):
    p1 = letter_probs(tok, model, *build_prompt(inst, qname, reverse=False))
    p2 = letter_probs(tok, model, *build_prompt(inst, qname, reverse=True))
    return {k: (p1[k] + p2[k]) / 2 for k in p1}   # average over the two option orders

def run_model(name, out_csv):
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float16).to("cuda").eval()
    rows, t0 = [], time.time()
    for k, inst in enumerate(instances):
        row = {x: inst[x] for x in ["dev_id", "type", "topic", "version", "gold_action", "gold_supported",
                                    "gold_specific", "gold_emergency", "gold_tool"]}
        for qname in QUESTIONS:
            p = avg_probs(tok, model, inst, qname)
            if any(v != v for v in p.values()):
                raise RuntimeError(f"NaN at {k}/{qname}: reload in float32")
            best = max(p, key=p.get)
            row[f"pred_{qname}"], row[f"conf_{qname}"] = best, p[best]
            row[f"probs_{qname}"] = json.dumps(p, ensure_ascii=False)
        rows.append(row)
        if k == 0:
            print(build_prompt(inst, "emergency", reverse=True)[0][1]["content"][-500:]); print(row)
    el = time.time() - t0
    print(f"{name}: {el:.0f}s, {1000*el/len(instances):.0f} ms/instance, peak GB {torch.cuda.max_memory_allocated()/1e9:.2f}")
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    del model; gc.collect(); torch.cuda.empty_cache()


In [ ]:
run_model("Qwen/Qwen2.5-0.5B-Instruct", "/kaggle/working/results_qwen0.5b_avg.csv")
run_model("Qwen/Qwen2.5-1.5B-Instruct", "/kaggle/working/results_qwen1.5b_avg.csv")

In [ ]:
import base64
from IPython.display import HTML, display

def dl(path):
    data = base64.b64encode(open(path, "rb").read()).decode()
    name = path.split("/")[-1]
    display(HTML(f'<a download="{name}" href="data:text/csv;base64,{data}">Download {name}</a>'))

dl("/kaggle/working/results_qwen0.5b_avg.csv")
dl("/kaggle/working/results_qwen1.5b_avg.csv")